In [1]:
import os
import tiktoken

parent_directory = os.path.abspath(os.path.join(os.getcwd(), os.pardir))

with open(os.path.join(parent_directory, "the-verdict.txt") , 'r', encoding="utf-8") as f:
    raw_text = f.read()

In [2]:
tokenizer = tiktoken.get_encoding("gpt2")
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [3]:
enc_sample = enc_text[50:] # remove first  50 tokens

context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"{x}")
print(f"    {y}")

[290, 4920, 2241, 287]
    [4920, 2241, 287, 257]


In [4]:
# demonstrating encoded next word prediction
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(f"{context} ----> {desired}")

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [5]:
# decoded version
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(f"{tokenizer.decode(context)} ----> {tokenizer.decode([desired])}")

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
# dataset for batched inputs and targets
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer,max_length, stride): # data loading
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i: i + max_length]
            target_chunk = token_ids[i+1,i + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

    def __len__(self):
        return len(self.input_ids)

ValueError: libcudnn.so.*[0-9] not found in the system path ['/home/brainybrigade/.local/share/uv/python/cpython-3.11-linux-x86_64-gnu/lib/python311.zip', '/home/brainybrigade/.local/share/uv/python/cpython-3.11-linux-x86_64-gnu/lib/python3.11', '/home/brainybrigade/.local/share/uv/python/cpython-3.11-linux-x86_64-gnu/lib/python3.11/lib-dynload', '', '/home/brainybrigade/Documents/Projects/LLM-from-Scratch/.venv/lib/python3.11/site-packages']